# Attention Alternatives: Sparse, Linear, and Recurrent

The full code of [Attention Alternatives: Sparse, Linear, and Recurrent](https://residual-stream.lvtan71.workers.dev/posts/attention-alternatives/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/attention-alternatives) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/attention-alternatives.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.
order_timing.py times on whatever GPU this machine has (the post's run used the M4's); the other scripts are arithmetic or numpy.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 0 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q numpy "torch==2.14.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "attention-alternatives"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/attention-alternatives


## `bills.py`

Prefill FLOPs, the KV cache of a Llama-3-8B shape, the linear-attention state, the n/d saving and a 4,096-token sliding window, as arithmetic.

In [2]:
!{sys.executable} bills.py

prefill FLOPs per head, d = 128
  n =     4,096         8.6 GFLOPs
  n =    32,768       549.8 GFLOPs
  n =   131,072     8,796.1 GFLOPs
  32x the tokens costs 1,024x the work

KV cache per token: 128 KiB
    context   KV cache  linear-attention state
      4,096    0.54 GB                  8.4 MB
     32,768    4.29 GB                  8.4 MB
    131,072   17.18 GB                  8.4 MB
  1,048,576  137.44 GB                  8.4 MB
20 sequences of 131,072 tokens: 344 GB = 4.3 H100s of 80 GB

score matrix at n = 32,768: 2.1 GB per head in BF16

linear attention: (QK^T)V costs 4n^2d, Q(K^T V) costs 4nd^2, ratio n/d
  n =      64       0.5x
  n =     128       1.0x
  n =   4,096      32.0x
  n = 131,072   1,024.0x

sliding window W = 4,096: cache capped at W tokens
  at n = 32,768: cache 8x smaller, 23% of the causal pairs computed; 32 layers reach 131,072 tokens


The post printed:

```text
prefill FLOPs per head, d = 128
  n =     4,096         8.6 GFLOPs
  n =    32,768       549.8 GFLOPs
  n =   131,072     8,796.1 GFLOPs
  32x the tokens costs 1,024x the work

KV cache per token: 128 KiB
    context   KV cache  linear-attention state
      4,096    0.54 GB                  8.4 MB
     32,768    4.29 GB                  8.4 MB
    131,072   17.18 GB                  8.4 MB
  1,048,576  137.44 GB                  8.4 MB
20 sequences of 131,072 tokens: 344 GB = 4.3 H100s of 80 GB

score matrix at n = 32,768: 2.1 GB per head in BF16

linear attention: (QK^T)V costs 4n^2d, Q(K^T V) costs 4nd^2, ratio n/d
  n =      64       0.5x
  n =     128       1.0x
  n =   4,096      32.0x
  n = 131,072   1,024.0x

sliding window W = 4,096: cache capped at W tokens
  at n = 32,768: cache 8x smaller, 23% of the causal pairs computed; 32 layers reach 131,072 tokens
```

## `linear_attention.py`

Linear attention computed as an n x n product, as a reassociated d x d product and as a recurrence, checked equal; and the rank of softmax's similarity matrix.

In [3]:
!{sys.executable} linear_attention.py

n = 512, d = 64


non-causal, quadratic vs linear:   max difference 1.7e-16
causal, quadratic vs recurrent:    max difference 6.7e-16


rank of exp(QK^T / sqrt d): 512 of 512


rank of phi(Q) phi(K)^T:    64 of 512


The post printed:

```text
n = 512, d = 64
non-causal, quadratic vs linear:   max difference 2.8e-16
causal, quadratic vs recurrent:    max difference 8.9e-16
rank of exp(QK^T / sqrt d): 512 of 512
rank of phi(Q) phi(K)^T:    64 of 512
```

## `order_timing.py`

Times (QK^T)V against Q(K^T V) from 32 to 16,384 tokens.

In [4]:
!{sys.executable} order_timing.py

device cuda, float32, d = 128, best of 20 runs
      n    (QK^T)V   Q(K^T V)  measured  counted


     32    0.041ms    0.042ms      1.0x    0.25x
     64    0.040ms    0.041ms      1.0x    0.50x
    128    0.041ms    0.041ms      1.0x    1.00x
    256    0.047ms    0.045ms      1.0x    2.00x


    512    0.049ms    0.048ms      1.0x    4.00x
  1,024    0.069ms    0.049ms      1.4x    8.00x
  2,048    0.176ms    0.049ms      3.6x   16.00x
  4,096    0.464ms    0.055ms      8.4x   32.00x


  8,192    1.607ms    0.073ms     22.1x   64.00x


 16,384    7.180ms    0.127ms     56.4x  128.00x


The post printed:

```text
device mps, float32, d = 128, best of 20 runs
      n    (QK^T)V   Q(K^T V)  measured  counted
     32    0.181ms    0.223ms      0.8x    0.25x
     64    0.180ms    0.186ms      1.0x    0.50x
    128    0.199ms    0.188ms      1.1x    1.00x
    256    0.227ms    0.200ms      1.1x    2.00x
    512    0.258ms    0.223ms      1.2x    4.00x
  1,024    0.441ms    0.233ms      1.9x    8.00x
  2,048    0.993ms    0.279ms      3.6x   16.00x
  4,096    3.006ms    0.373ms      8.1x   32.00x
  8,192   11.809ms    0.467ms     25.3x   64.00x
 16,384   51.872ms    0.688ms     75.4x  128.00x
```

## `recall.py`

Writes n random key-value pairs into a KV cache, a linear-attention state and a delta-rule state, and reads each back: by n, by state width, and by when each pair was written.

In [5]:
!{sys.executable} recall.py

key and value width d = 64; the state is d x d = 4,096 numbers
 pairs  cache size        cache       linear        delta
         (numbers)  correct cos  correct cos  correct cos
    16       2,048   100%  1.00   100%  0.91   100%  0.95
    32       4,096   100%  1.00   100%  0.81   100%  0.84
    64       8,192   100%  1.00   100%  0.71    88%  0.67


   128      16,384   100%  1.00    96%  0.59    59%  0.46
   256      32,768   100%  1.00    70%  0.44    27%  0.25
   512      65,536   100%  1.00    35%  0.33    12%  0.13


 1,024     131,072   100%  1.00     9%  0.24     7%  0.07


 4,096     524,288   100%  1.00     0%  0.12     2%  0.02

state size dial: 256 pairs, linear and delta memories of width d
    d    state       linear        delta


   32    1,024    16%  0.34    11%  0.13
   64    4,096    73%  0.45    25%  0.25
  128   16,384   100%  0.57    62%  0.43


  256   65,536   100%  0.71   100%  0.68


  512  262,144   100%  0.82   100%  0.84

256 pairs, d = 64: share read back correctly, by when the pair was written
       written  linear   delta
          1-32     84%      0%
         33-64     75%      0%
         65-96     81%      0%
        97-128     75%      0%
       129-160     66%      3%
       161-192     84%     22%
       193-224     84%     81%
       225-256     84%    100%

key 0 written with value a, 31 other pairs, then key 0 again with value b
  linear reads key 0: cos to a  0.73, cos to b  0.71
  delta  reads key 0: cos to a  0.21, cos to b  1.00


The post printed:

```text
key and value width d = 64; the state is d x d = 4,096 numbers
 pairs  cache size        cache       linear        delta
         (numbers)  correct cos  correct cos  correct cos
    16       2,048   100%  1.00   100%  0.91   100%  0.95
    32       4,096   100%  1.00   100%  0.81   100%  0.84
    64       8,192   100%  1.00   100%  0.71    88%  0.67
   128      16,384   100%  1.00    96%  0.59    59%  0.46
   256      32,768   100%  1.00    70%  0.44    27%  0.25
   512      65,536   100%  1.00    35%  0.33    12%  0.13
 1,024     131,072   100%  1.00     9%  0.24     7%  0.07
 4,096     524,288   100%  1.00     0%  0.12     2%  0.02

state size dial: 256 pairs, linear and delta memories of width d
    d    state       linear        delta
   32    1,024    16%  0.34    11%  0.13
   64    4,096    73%  0.45    25%  0.25
  128   16,384   100%  0.57    62%  0.43
  256   65,536   100%  0.71   100%  0.68
  512  262,144   100%  0.82   100%  0.84

256 pairs, d = 64: share read back correctly, by when the pair was written
       written  linear   delta
          1-32     84%      0%
         33-64     75%      0%
         65-96     81%      0%
        97-128     75%      0%
       129-160     66%      3%
       161-192     84%     22%
       193-224     84%     81%
       225-256     84%    100%

key 0 written with value a, 31 other pairs, then key 0 again with value b
  linear reads key 0: cos to a  0.73, cos to b  0.71
  delta  reads key 0: cos to a  0.21, cos to b  1.00
```

## `ssm.py`

A diagonal state space model: discretization, recurrence against convolution, input-dependent parameters, a parallel scan, and Mamba-2's quadratic form.

In [6]:
!{sys.executable} ssm.py

delta is a gate: Abar = exp(delta A) for this model's slowest and fastest state
  delta = 0.001   keeps 1.000 and 0.996 of the old state
  delta = 0.1     keeps 0.990 and 0.692 of the old state
  delta = 10.0    keeps 0.376 and 0.000 of the old state

fixed parameters, recurrence vs convolution: max difference 2.2e-15
input-dependent, recurrence vs first token's kernel: max difference 4.9e+00
parallel scan in 8 steps vs 256 sequential steps: max difference 4.0e-15


scalar decay, linear recurrence vs quadratic masked form: max difference 1.4e-13
with every a_t = 1, L is the causal mask and (L * (C B^T)) x is causal linear attention, Q = C, K = B, V = x


The post printed:

```text
delta is a gate: Abar = exp(delta A) for this model's slowest and fastest state
  delta = 0.001   keeps 1.000 and 0.996 of the old state
  delta = 0.1     keeps 0.990 and 0.692 of the old state
  delta = 10.0    keeps 0.376 and 0.000 of the old state

fixed parameters, recurrence vs convolution: max difference 2.9e-15
input-dependent, recurrence vs first token's kernel: max difference 4.9e+00
parallel scan in 8 steps vs 256 sequential steps: max difference 3.6e-15
scalar decay, linear recurrence vs quadratic masked form: max difference 1.4e-13
with every a_t = 1, L is the causal mask and (L * (C B^T)) x is causal linear attention, Q = C, K = B, V = x
```

## `configs.py`

Reads the attention layout of thirteen released models from their config.json at pinned revisions, and the KV cache their attention layers need at 256K tokens.

In [7]:
!{sys.executable} configs.py

full attention, for comparison


Llama-2-7b-hf           32 of  32 layers attention (100%), KV cache at 256K  128.0 GiB (all-attention  128.0 GiB)


Mixtral-8x7B-v0.1       32 of  32 layers attention (100%), KV cache at 256K   32.0 GiB (all-attention   32.0 GiB)

sliding windows


Mistral-7B-v0.1        sliding_window 4096 in all 32 layers


gpt-oss-20b            12 sliding (128 tokens) and 12 full layers, alternating: ['sliding_attention', 'full_attention', 'sliding_attention', 'full_attention']

hybrids: attention beside linear-attention or state-space layers


...

MiniMax-Text-01         10 of  80 layers attention ( 12%), KV cache at 256K   10.0 GiB (all-attention   80.0 GiB); the rest lightning (linear) attention

back to full attention, then sparse


MiniMax-M2              62 of  62 layers attention (100%), KV cache at 256K   62.0 GiB (all-attention   62.0 GiB)


MiniMax-M3             57 of 60 layers sparse: top 16 blocks of 128 tokens per query, 2,048 keys; the first 3 layers full

learned sparse attention


DeepSeek-V3.2-Exp      indexer picks top 2,048 earlier tokens per query, in all 61 layers


DeepSeek-V4-Pro        61 layers: 30 merge 4 tokens per entry (CSA), 31 merge 128 (HCA); top 1,024 entries, window 128


The post printed:

```text
full attention, for comparison
Llama-2-7b-hf           32 of  32 layers attention (100%), KV cache at 256K  128.0 GiB (all-attention  128.0 GiB)
Mixtral-8x7B-v0.1       32 of  32 layers attention (100%), KV cache at 256K   32.0 GiB (all-attention   32.0 GiB)

sliding windows
Mistral-7B-v0.1        sliding_window 4096 in all 32 layers
gpt-oss-20b            12 sliding (128 tokens) and 12 full layers, alternating: ['sliding_attention', 'full_attention', 'sliding_attention', 'full_attention']

hybrids: attention beside linear-attention or state-space layers
Jamba-v0.1               4 of  32 layers attention ( 12%), KV cache at 256K    4.0 GiB (all-attention   32.0 GiB); Mamba state size 16
Nemotron-H-8B            4 of  28 mixers attention ( 14%), KV cache at 256K    4.0 GiB (all-attention   28.0 GiB); plus 24 MLP layers
Qwen3-Next-80B-A3B      12 of  48 layers attention ( 25%), KV cache at 256K    6.0 GiB (all-attention   24.0 GiB); one attention layer every 4, the rest Gated DeltaNet
Kimi-Linear-48B-A3B      7 of  27 layers attention ( 26%); attention (MLA) at layers [4, 8, 12, 16, 20, 24, 27], Kimi Delta Attention elsewhere
MiniMax-Text-01         10 of  80 layers attention ( 12%), KV cache at 256K   10.0 GiB (all-attention   80.0 GiB); the rest lightning (linear) attention

back to full attention, then sparse
MiniMax-M2              62 of  62 layers attention (100%), KV cache at 256K   62.0 GiB (all-attention   62.0 GiB)
MiniMax-M3             57 of 60 layers sparse: top 16 blocks of 128 tokens per query, 2,048 keys; the first 3 layers full

learned sparse attention
DeepSeek-V3.2-Exp      indexer picks top 2,048 earlier tokens per query, in all 61 layers
DeepSeek-V4-Pro        61 layers: 30 merge 4 tokens per entry (CSA), 31 merge 128 (HCA); top 1,024 entries, window 128
```